### One important interview takeaway
The watermark must be advanced only after the target load succeeds. Otherwise, a failed load could cause records to be skipped on the next run.

### The complete architecture we're implementing

                  ADLS
                    │
                    ▼
              Bronze Customers
                    │
                    │
             Read watermark
                    │
                    ▼
       updated_at > last_watermark
                    │
                    ▼
          Incremental Records
                    │
             ┌──────┴──────┐
             │             │
           Valid         Invalid
             │             │
             │             ▼
             │       Quarantine
             │
             ▼
          Dedup
             │
             ▼
        MERGE into Silver
          │           │
       UPDATE       INSERT
          │           │
          └─────┬─────┘
                │
                ▼
       Calculate new watermark
                │
                ▼
       Update control table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbx_fintech_data_platform.metadata.watermark_control (
    pipeline_name STRING,
    source_table STRING,
    target_table STRING,
    watermark_column STRING,
    last_watermark TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA;

In [0]:
%sql
ALTER TABLE dbx_fintech_data_platform.metadata.watermark_control
SET TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name'
);

In [0]:
%sql
ALTER TABLE dbx_fintech_data_platform.metadata.watermark_control
RENAME COLUMN updated_at TO watermark_updated_at;

In [0]:
%sql
SELECT * 
FROM dbx_fintech_data_platform.metadata.watermark_control;

In [0]:
%sql
SELECT
    MAX(updated_at) AS max_updated_at
FROM dbx_fintech_data_platform.bronze.customers
WHERE _source_date = '2026-08-18';

In [0]:
%sql
INSERT INTO dbx_fintech_data_platform.metadata.watermark_control
SELECT
    'customer_incremental' AS pipeline_name,
    'dbx_fintech_data_platform.bronze.customers' AS source_table,
    'dbx_fintech_data_platform.silver.customers' AS target_table,
    'updated_at' AS watermark_column,
    MAX(updated_at) AS last_watermark,
    current_timestamp() AS watermark_updated_at
FROM dbx_fintech_data_platform.bronze.customers
WHERE _source_date = '2026-08-18';

In [0]:
%sql
SELECT *
FROM dbx_fintech_data_platform.metadata.watermark_control;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW customer_incremental_test AS

SELECT *
FROM VALUES
    ('TEST_001', 'Before Watermark', 'before@test.com', '9000000001', 'IN', 'STANDARD',
     TIMESTAMP('2026-07-30 10:00:00'), TIMESTAMP('2026-07-30 10:00:00')),

    ('TEST_002', 'Before Watermark 2', 'before2@test.com', '9000000002', 'IN', 'STANDARD',
     TIMESTAMP('2026-07-31 20:00:00'), TIMESTAMP('2026-07-31 20:00:00')),

    ('TEST_003', 'Updated Customer', 'updated@test.com', '9000000003', 'IN', 'PREMIUM',
     TIMESTAMP('2026-08-01 10:00:00'), TIMESTAMP('2026-08-01 10:00:00')),

    ('TEST_004', 'Updated Customer 2', 'updated2@test.com', '9000000004', 'US', 'VIP',
     TIMESTAMP('2026-08-02 12:00:00'), TIMESTAMP('2026-08-02 12:00:00')),

    ('TEST_005', 'New Customer', 'new@test.com', '9000000005', 'UK', 'STANDARD',
     TIMESTAMP('2026-08-03 09:00:00'), TIMESTAMP('2026-08-03 09:00:00'))

AS t(
    customer_id,
    name,
    email,
    phone,
    country,
    customer_type,
    created_at,
    updated_at
);

In [0]:
%sql
SELECT *
FROM customer_incremental_test
ORDER BY updated_at;

In [0]:
%sql
SELECT *
FROM customer_incremental_test
WHERE updated_at > (
    SELECT last_watermark
    FROM dbx_fintech_data_platform.metadata.watermark_control
    WHERE pipeline_name = 'customer_incremental'
);

In [0]:
%sql

SELECT
    pipeline_name,
    source_table,
    target_table,
    watermark_column,
    last_watermark,
    watermark_updated_at
FROM dbx_fintech_data_platform.metadata.watermark_control
WHERE pipeline_name = 'customer_incremental';

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW customer_incremental_source AS

SELECT *
FROM dbx_fintech_data_platform.bronze.customers
WHERE updated_at > (
    SELECT last_watermark
    FROM dbx_fintech_data_platform.metadata.watermark_control
    WHERE pipeline_name = 'customer_incremental'
);

In [0]:
%sql

SELECT
    COUNT(*) AS incremental_records,
    MIN(updated_at) AS min_updated_at,
    MAX(updated_at) AS max_updated_at
FROM customer_incremental_source;

In [0]:
incremental_count = spark.sql("""
    SELECT COUNT(*) AS cnt
    FROM customer_incremental_source
""").collect()[0]["cnt"]

print(f"Incremental records found: {incremental_count}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

incremental_df = spark.table("customer_incremental_source")

clean_df = (
    incremental_df

    # Standardization
    .withColumn("name", F.trim(F.col("name")))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("country", F.upper(F.trim(F.col("country"))))
    .withColumn("customer_type", F.upper(F.trim(F.col("customer_type"))))
    .withColumn("phone", F.col("phone").cast("string"))

    # Timestamp normalization
    .withColumn("created_at", F.col("created_at").cast("timestamp"))
    .withColumn("updated_at", F.col("updated_at").cast("timestamp"))
)

In [0]:
valid_df = clean_df.filter(
    (F.col("customer_id").isNotNull()) &
    (F.trim(F.col("customer_id")) != "") &

    (F.col("email").isNotNull()) &
    (F.trim(F.col("email")) != "") &

    (F.col("country").isin("CA", "DE", "IN", "UK", "US")) &

    (F.col("customer_type").isin("STANDARD", "PREMIUM", "VIP"))
)

In [0]:
invalid_df = clean_df.filter(
    ~(
        (F.col("customer_id").isNotNull()) &
        (F.trim(F.col("customer_id")) != "") &

        (F.col("email").isNotNull()) &
        (F.trim(F.col("email")) != "") &

        (F.col("country").isin("CA", "DE", "IN", "UK", "US")) &

        (F.col("customer_type").isin("STANDARD", "PREMIUM", "VIP"))
    )
)

In [0]:
quarantine_df = (
    invalid_df
    .withColumn(
        "quarantine_reason",
        F.lit("Incremental customer DQ validation failed")
    )
    .withColumn(
        "quarantined_at",
        F.current_timestamp()
    )
)

In [0]:
if quarantine_df.limit(1).count() > 0:

    (
        quarantine_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(
            "dbx_fintech_data_platform.silver.customers_quarantine"
        )
    )

    print("Invalid records quarantined.")

else:
    print("No invalid records to quarantine.")

In [0]:
window_spec = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("updated_at").desc(),
        F.col("_source_date").desc()
    )
)

incremental_latest_df = (
    valid_df
    .withColumn("_rn", F.row_number().over(window_spec))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

In [0]:
print(
    f"Records after incremental DQ + dedup: "
    f"{incremental_latest_df.count()}"
)

In [0]:
incremental_latest_df.createOrReplaceTempView(
    "customer_incremental_ready"
)



In [0]:
%sql


MERGE INTO dbx_fintech_data_platform.silver.customers AS target

USING customer_incremental_ready AS source

ON target.customer_id = source.customer_id

WHEN MATCHED
AND source.updated_at > target.updated_at
THEN UPDATE SET
    target.name = source.name,
    target.email = source.email,
    target.phone = source.phone,
    target.country = source.country,
    target.customer_type = source.customer_type,
    target.created_at = source.created_at,
    target.updated_at = source.updated_at

WHEN NOT MATCHED
THEN INSERT (
    customer_id,
    name,
    email,
    phone,
    country,
    customer_type,
    created_at,
    updated_at
)
VALUES (
    source.customer_id,
    source.name,
    source.email,
    source.phone,
    source.country,
    source.customer_type,
    source.created_at,
    source.updated_at
);

In [0]:
new_watermark = spark.sql("""
    SELECT MAX(updated_at) AS max_updated_at
    FROM customer_incremental_ready
""").collect()[0]["max_updated_at"]

print(f"New watermark candidate: {new_watermark}")

In [0]:
if new_watermark is not None:

    spark.sql(f"""
        UPDATE dbx_fintech_data_platform.metadata.watermark_control
        SET
            last_watermark = TIMESTAMP('{new_watermark}'),
            watermark_updated_at = current_timestamp()
        WHERE pipeline_name = 'customer_incremental'
    """)

    print(f"Watermark updated to: {new_watermark}")

else:

    print("No new data. Watermark remains unchanged.")

In [0]:
%sql

SELECT
    pipeline_name,
    watermark_column,
    last_watermark,
    watermark_updated_at
FROM dbx_fintech_data_platform.metadata.watermark_control
WHERE pipeline_name = 'customer_incremental';

In [0]:
%sql

SELECT
    COUNT(*) AS total_customers,
    COUNT(DISTINCT customer_id) AS unique_customers,
    MIN(updated_at) AS min_updated_at,
    MAX(updated_at) AS max_updated_at
FROM dbx_fintech_data_platform.silver.customers;